<a href="https://colab.research.google.com/github/yuhan222/114-2-Programing-Language/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip uninstall -y google-genai google-generativeai && pip install -q -U google-generativeai

Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6


In [ ]:
import gradio as gr
import pandas as pd
from google.colab import auth, userdata
from google.auth import default
import gspread
from datetime import datetime
import time

# 使用經典穩定版 SDK
import google.generativeai as genai
from google.generativeai.types import HarmCategory, HarmBlockThreshold

# --- 1. 身分驗證 ---
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# --- 2. 初始化 AI ---
try:
    key = userdata.get('gemini')
    genai.configure(api_key=key)
    model = genai.GenerativeModel(
        model_name='gemini-2.5-flash-lite',
        safety_settings={
            HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
        }
    )
except Exception as e:
    print(f"❌ 初始化失敗: {e}")

SHEET_URL = "https://docs.google.com/spreadsheets/d/1xBFqAnS-W1WzXEqhYGcMkoCYZ295mOI_j94tppr-H2o/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"
HEADERS = ["日期時間", "科目", "成績", "AI 建議"]

# --- 讀取函式 ---
def load_all_data_from_sheet():
    try:
        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)
        all_data = ws.get_all_values()
        if len(all_data) > 1:
            df = pd.DataFrame(all_data[1:], columns=all_data[0])
            # 確保成績欄位是數字型態，這對畫圖非常重要
            df['成績'] = pd.to_numeric(df['成績'], errors='coerce')
            return df
        return pd.DataFrame(columns=HEADERS)
    except:
        return pd.DataFrame(columns=HEADERS)

# --- 3. 核心功能：新增至暫存區 ---
def add_to_staging(subject, grade, current_list):
    if not subject: return current_list, current_list, gr.update()
    current_list.append([subject, grade])
    return current_list, current_list, 0

# --- 4. 核心功能：批次處理並寫入 ---
def batch_process_and_save(staging_list):
    if not staging_list or len(staging_list) == 0:
        return "⚠️ 待處理列表是空的，請先新增成績！", staging_list, staging_list, load_all_data_from_sheet(), gr.update()

    results_log = []
    try:
        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)
        if not ws.get_all_values(): ws.append_row(HEADERS)

        for item in staging_list:
            subject, grade = item[0], item[1]
            summary = "（AI 暫時無法回應）"
            prompt_text = f"你是一位親切的程式老師。學生在 {subject} 拿到了 {grade} 分。請用繁體中文給予一句鼓勵與學習建議。"

            try:
                response = model.generate_content(prompt_text)
                if response and response.text: summary = response.text
            except Exception as e:
                pass

            today = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            ws.insert_row([today, subject, str(grade), summary], index=2)
            results_log.append(f"✅ {subject} ({grade}分) 已記錄！")
            time.sleep(3)

        final_message = "🎉 批次處理完畢！\n" + "\n".join(results_log)
        empty_list = []
        updated_df = load_all_data_from_sheet()

        # 寫入完畢後，回傳更新的 DataFrame，並且更新下拉選單的選項（抓取最新的所有科目）
        unique_subjects = updated_df['科目'].dropna().unique().tolist()
        dropdown_update = gr.update(choices=unique_subjects) if unique_subjects else gr.update()

        return final_message, empty_list, empty_list, updated_df, dropdown_update

    except Exception as e:
        return f"❌ 寫入失敗：{str(e)}", staging_list, staging_list, load_all_data_from_sheet(), gr.update()

# --- 5. 新增功能：畫分科趨勢圖 ---
def plot_subject_trend(subject, df):
    if subject is None or df.empty:
        return None # 如果沒選科目或沒資料，就不畫圖

    # 1. 篩選出該科目的資料
    subject_df = df[df['科目'] == subject].copy()

    if subject_df.empty:
        return None

    # 2. 轉換日期格式以便排序
    subject_df['日期時間'] = pd.to_datetime(subject_df['日期時間'], errors='coerce')

    # 3. 按照時間先後排序 (畫圖時時間軸由左至右)
    subject_df = subject_df.sort_values(by='日期時間')

    # 4. 把日期轉回易讀的字串，供 X 軸顯示
    subject_df['日期顯示'] = subject_df['日期時間'].dt.strftime('%m-%d %H:%M')

    # 回傳篩選過且排序好的 DataFrame，Gradio 的 LinePlot 會接手畫出來
    return subject_df

# --- 6. 建立介面 ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎓 學生作業成績 AI 記錄系統 (批次處理 + 趨勢圖版)")
    staging_state = gr.State([])

    # 啟動時先載入一次資料，供下方使用
    initial_df = load_all_data_from_sheet()
    initial_subjects = initial_df['科目'].dropna().unique().tolist() if not initial_df.empty else []

    with gr.Tabs():
        # --- 分頁 1：成績輸入與歷史紀錄 ---
        with gr.TabItem("📝 成績記錄區"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 1. 新增成績至列表")
                    input_subject = gr.Dropdown(label="科目", choices=["數學", "國文", "英文", "生物", "物理", "化學"], allow_custom_value=True)
                    input_grade = gr.Number(label="成績", value=0)
                    btn_add = gr.Button("➕ 加入待處理列表", variant="secondary")

                with gr.Column(scale=2):
                    gr.Markdown("### 2. 待處理列表")
                    staging_table = gr.DataFrame(headers=["科目", "成績"], datatype=["str", "number"], interactive=False)
                    btn_submit_batch = gr.Button("🚀 批次送出並生成 AI 建議", variant="primary")

            output_text = gr.Textbox(label="處理進度與結果", lines=4)
            output_table = gr.DataFrame(label="所有歷史成績記錄", value=initial_df)

        # --- 分頁 2：成績趨勢分析 ---
        with gr.TabItem("📈 成績趨勢圖"):
            gr.Markdown("請選擇科目以查看歷史成績起伏趨勢。")
            with gr.Row():
                # 用來選擇科目的下拉選單，會自動抓取歷史紀錄中出現過的科目
                plot_subject_dropdown = gr.Dropdown(
                    label="選擇分析科目",
                    choices=initial_subjects,
                    value=initial_subjects[0] if initial_subjects else None
                )

            # 折線圖元件
            trend_plot = gr.LinePlot(
                x="日期顯示",
                y="成績",
                title="成績變化趨勢",
                tooltip=["日期顯示", "成績"],
                width=800,
                height=400,
                y_lim=[0, 100]
            )

    # --- 綁定事件 ---
    # 1. 加入清單
    btn_add.click(fn=add_to_staging, inputs=[input_subject, input_grade, staging_state], outputs=[staging_state, staging_table, input_grade])

    # 2. 批次送出 (注意：這邊會連帶更新圖表選單的科目清單)
    btn_submit_batch.click(
        fn=batch_process_and_save,
        inputs=[staging_state],
        outputs=[output_text, staging_state, staging_table, output_table, plot_subject_dropdown]
    )

    # 3. 畫圖觸發器 (當下拉選單改變，或是歷史表格有更新時，都會重畫圖表)
    plot_subject_dropdown.change(
        fn=plot_subject_trend,
        inputs=[plot_subject_dropdown, output_table],
        outputs=[trend_plot]
    )

    # 初次載入或表格更新時，若有預設科目，也自動畫一次圖
    output_table.change(
        fn=plot_subject_trend,
        inputs=[plot_subject_dropdown, output_table],
        outputs=[trend_plot]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_7497/3575300225.py:123: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3743a6a24e76ee5e51.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
